In [2]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from tqdm import tqdm

In [36]:
in_data = "./DATASET"
out_data = "./csvs3/"
os.makedirs(out_data, exist_ok=True)

## Current Threshold: redefining/restricting operational zone
c_thresh = 4300

## Thresholding distance to preceeding point to detemrine continuous scan
delta_thresh = 0.035

## Thresholding number of points to identify unexposed layer
unexposed_thresh = 10000

xa_max = -27.69558334350586
xb_min = -27.11699104309082
xc_max = -19.144311904907227
xd_min = -18.56609344482422

ya_min = -80.22529602050781
yb_max = -80.80426788330078
yc_min = -88.77432250976562
yd_max = -89.3532943725586

xb_max = -26.98001480102539
xc_min = -19.24423599243164
yb_min = -80.96295166015625
yc_max = -88.69274139404297

## Looping through each layer and creating preprocessed csvs
for f in tqdm(os.listdir(in_data)):
    in_path = os.path.join(in_data, f)
    if os.path.exists(in_path) and os.path.basename(in_path).endswith('.hdf5'):
        tag = f.split(".")[0]
        
        out_path = os.path.join(out_data, tag + ".csv")
        
        file_info = h5py.File(in_path, 'r')
        file_data = file_info['OpenData']
        x = file_data[0]
        y = file_data[1]
        power = file_data[2]
        speed = file_data[3]
        dia = file_data[4]
        cur = file_data[5]
        sig = file_data[6]
        id1 = file_data[7]
        id2 = file_data[8]
        c1 = file_data[9]
        c2 = file_data[10]

        ## N x 11 datapoints per layer
        data = np.vstack([x, y, power, speed, dia, cur, sig, id1, id2, c1, c2]).T

        ## Filter out where the current is not in operational zone
        opzone_condition = data[:, 5] > c_thresh
        data = data[opzone_condition]
        
        ## Determine continuous scans and grouping with scan numbers
        x = data[:, 0]
        y = data[:, 1]
        is_bulk = data[:, 7]
        deltas = np.zeros_like(x)
        scan_nums = np.zeros_like(x, dtype=int)
        in_control = np.zeros_like(x, dtype=int)
        scan = int(0)
        
        
        for i in range(1, len(deltas)):
            deltas[i] = np.sqrt((x[i]-x[i-1])**2 + (y[i]-y[i-1])**2)
            if deltas[i] > delta_thresh:
                scan+= int(1)
            scan_nums[i] = scan 

            if is_bulk[i] == 1:
                in_control[i] = 1

            else:
                if len(deltas) < unexposed_thresh:
                    in_control[i] = 1

                elif (xa_max <= x[i] <= xb_min) or (xc_max <= x[i] <= xd_min) or (ya_min >= y[i] >= yb_max) or (yc_min >= y[i] >= yd_max):
                    in_control[i] = 1

                elif (xb_max < x[i] < xc_min) and (yb_min > y[i] > yc_max):
                    in_control[i] = 0
                else:
                    in_control[i] = -1

            
        ## Expanding data array to include euclidean distance from previous point and scan number
        deltas = np.expand_dims(deltas, -1)
        scan_nums = np.expand_dims(scan_nums, -1).astype(int)
        in_control = np.expand_dims(in_control, -1).astype(int)
        data = np.hstack([data, deltas, scan_nums, in_control])

        

        # if len(x) < 10000:
        #     print(tag)
        #     ys_for_center_xs = y[(-26 < x) & (x < -20)]
        #     mid_y = (ys_for_center_xs.max() + ys_for_center_xs.min())/2
        #     print("mid_y", mid_y)
        #     top_for_center_xs = ys_for_center_xs[ys_for_center_xs > mid_y] 
        #     bottom_for_center_xs = ys_for_center_xs[ys_for_center_xs < mid_y]
            
        #     xs_for_center_ys = x[(-88 < y) & (y < -82)]
        #     mid_x = (xs_for_center_ys.max() + xs_for_center_ys.min())/2
        #     left_for_center_ys = xs_for_center_ys[xs_for_center_ys < mid_x]
        #     xa = left_for_center_ys.min()
        #     xb = left_for_center_ys.max()
        #     right_for_center_ys = xs_for_center_ys[xs_for_center_ys > mid_x]
        #     xc = right_for_center_ys.min()
        #     xd = right_for_center_ys.max()
            
        #     # print("top_for_center_xs")
        #     # print(top_for_center_xs)
        #     # print(top_for_center_xs.max())
        #     # print(top_for_center_xs.min())
        #     # print(top_for_center_xs.shape)
    
        #     # print("bottom_for_center_xs")
        #     # print(bottom_for_center_xs)
        #     # print(bottom_for_center_xs.max())
        #     # print(bottom_for_center_xs.min())
        #     # print(bottom_for_center_xs.shape)
        #     print("xa", xa)
        #     print("xb", xb)
        #     print("xc", xc)
        #     print("xd", xd)
        #     break

        ## Save to CSV
        headers = [
            "x", "y", "power", "speed", "spot_diameter", "laser_current", 
            "signal", "ID1", "ID2", "C1", "C2", "delta", "scan_number", "in_control"]
        df = pd.DataFrame(data, columns=headers)
        df = df[df["in_control"]> -1]
        df.to_csv(out_path)
    
        # break
# print(tag)
# df

100%|█████████████████████████████████| 381/381 [02:43<00:00,  2.32it/s]


In [ ]:
# ## Testing - Sorting Directory Contents + Visualizing scan numbers
# data_dir = "./csvs/"
# scan_imgs_dir = "./scan_imgs/"
# os.makedirs(scan_imgs_dir, exist_ok=True)

# files = os.listdir(data_dir)
# files = [f for f in files if f.endswith(".csv")]
# files.sort(key=lambda x: int(x.split(".")[0][5:]))

# ## Visualizing scan numbers
# hues = ["red", "blue", "green", "purple", "orange", "black", "grey", "brown"]
# for f_idx in tqdm(range(len(files))): 
#     file_path = os.path.join(data_dir, files[f_idx])
#     tag = files[f_idx].split(".")[0]
#     img_path = os.path.join(scan_imgs_dir, tag+".png")
#     data_df = pd.read_csv(file_path)

#     x = data_df["x"].to_numpy()
#     y = data_df["y"].to_numpy()
#     scan_nums = data_df["scan_number"].to_numpy()
#     coloring = []
#     for i in range(len(x)):
#         col_idx = int(scan_nums[i]) % len(hues)
#         coloring.append(hues[col_idx])

#     j=len(x)
#     # print(j)
#     plt.figure()
#     plt.scatter(x[:j], y[:j], c=coloring[:j], s =2)
#     plt.savefig(img_path, dpi=300)
#     plt.close()

In [37]:
## Testing - Sorting Directory Contents + Visualizing scan numbers
data_dir = "./csvs3/"
scan_imgs_dir = "./scan_imgs3/"
os.makedirs(scan_imgs_dir, exist_ok=True)

files = os.listdir(data_dir)
files = [f for f in files if f.endswith(".csv")]
files.sort(key=lambda x: int(x.split(".")[0][5:]))

## Visualizing scan numbers
hues = ["red", "blue", "green", "purple", "orange", "black", "grey"]
for f_idx in tqdm(range(len(files))): 
    file_path = os.path.join(data_dir, files[f_idx])
    tag = files[f_idx].split(".")[0]
    img_path = os.path.join(scan_imgs_dir, tag+".png")
    data_df = pd.read_csv(file_path)

    x = data_df["x"].to_numpy()
    y = data_df["y"].to_numpy()
    in_control = data_df["in_control"].to_numpy()
    coloring = []
    for i in range(len(x)):
        # col_idx = int(in_control[i]) % len(hues)
        coloring.append(hues[int(in_control[i])])

    j=len(x)
    # print(j)
    plt.figure()
    plt.scatter(x[:j], y[:j], c=coloring[:j], s =2)
    plt.savefig(img_path, dpi=300)
    plt.close()

100%|█████████████████████████████████| 379/379 [05:58<00:00,  1.06it/s]
